# Dense-grid cleanest test: actual grid vs ideal y-selector

**Core question:**
Does the real grid-on system produce the same polarization-angle pattern as an ideal y-selector
through the ordinary no-grid telescope?

**Constructions:**

| Variable | Formula | Meaning |
|----------|---------|--------|
| `S_actual` | `0.5*(S_x_on + S_y_on)` | Actual grid calibrator: dense grid + telescope, unpolarized input |
| `S_ideal`  | `0.5*S_y_off`            | Ideal y-selector: y-pol through the no-grid telescope, halved for unpolarized input |

**Core test:**
```
psi_actual = 0.5 * atan2(U_actual, Q_actual)
psi_ideal  = 0.5 * atan2(U_ideal,  Q_ideal)
delta_psi_grid_error = wrap(psi_actual - psi_ideal)  # to [-90, 90)
```

If the weighted mean and RMS of `delta_psi_grid_error` are small over the main beam,
the dense grid acts approximately as an ideal selector.
If the mean is small but the RMS or residual maps are structured,
the grid does not produce a large global angle bias but does modify local polarized beam structure.

> **Physical cautions:**
> - Stokes parameters are combined incoherently. Complex fields are NOT summed.
> - Polarization angle maps are unreliable where p is small. Always apply a dB mask.
> - y-pol angles live near 90 deg. Always use robust wrapped angle differences.
> - A small signed weighted mean can hide structured positive/negative lobes. Always check RMS and maps.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# ---------------------------------------------------------------
# Edit these paths to point to your GRASP .grd output files.
# All four files must share the same coordinate grid.
# ---------------------------------------------------------------
data_dir = "../data/wire_on_off/"

file_xoff = data_dir + "azel_1deg_offx.grd"    # no-grid, x input polarization
file_yoff = data_dir + "azel_1deg_offy.grd"    # no-grid, y input polarization
file_xon  = data_dir + "azel_1deg_densex.grd"  # dense-grid-on, x input polarization
file_yon  = data_dir + "azel_1deg_densey.grd"  # dense-grid-on, y input polarization

# dB threshold for statistics and plots (default: -20 dB relative to peak)
DB_DEFAULT = -20.0


## 1. Load four runs and compute Stokes

> **Cautionary notes:**
> - Stokes parameters are combined incoherently below. Do **not** add complex fields for unpolarized input.
> - `S_unpol_off = 0.5*(S_x_off + S_y_off)` is a diagnostic, not a calibration signal.
> - The ideal selector `S_ideal = 0.5*S_y_off` is the key reference.


In [ ]:
def load_grasp_grd(filepath):
    # Load GRASP .grd file. Returns x, y (1D), F1, F2 (2D complex, shape ny x nx).
    with open(filepath, 'r') as f:
        lines = f.readlines()
    grid_idx = None
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) == 3:
            try:
                nx, ny = int(parts[0]), int(parts[1])
                grid_idx = i
                break
            except Exception:
                continue
    if grid_idx is None:
        raise ValueError(f"Could not find grid size line in {filepath}")
    data = np.loadtxt(lines[grid_idx + 1:])
    F1 = (data[:, 0] + 1j * data[:, 1]).reshape(ny, nx)
    F2 = (data[:, 2] + 1j * data[:, 3]).reshape(ny, nx)
    for i in range(grid_idx):
        parts = lines[i].strip().split()
        if len(parts) == 4:
            try:
                xmin, ymin, xmax, ymax = map(float, parts)
            except Exception:
                continue
    x = np.linspace(xmin, xmax, nx)
    y = np.linspace(ymin, ymax, ny)
    return x, y, F1, F2


def compute_stokes(F1, F2):
    # Stokes I, Q, U, V from complex field components (linear basis).
    E1E1  = np.abs(F1)**2
    E2E2  = np.abs(F2)**2
    cross = F1 * np.conj(F2)
    I = E1E1 + E2E2
    Q = E1E1 - E2E2
    U = 2.0 * np.real(cross)
    V = -2.0 * np.imag(cross)
    return I, Q, U, V


def make_db_mask(I, db_min=-20):
    # True where I >= db_min dB relative to peak.
    I_db = 10.0 * np.log10(np.maximum(I, 1e-300) / np.nanmax(I))
    return I_db >= db_min


def pol_fraction(Q, U, I):
    return np.sqrt(Q**2 + U**2) / np.maximum(I, 1e-30)


def pol_angle_deg(Q, U):
    # Linear polarization angle: 0.5*atan2(U, Q) in degrees.
    return 0.5 * np.degrees(np.arctan2(U, Q))


def pol_angle_diff_deg(psi_a, psi_b):
    # Wrapped difference psi_a - psi_b in [-90, 90) deg.
    # Uses doubled-angle space to handle the 180-deg ambiguity correctly.
    two_a = np.radians(2.0 * psi_a)
    two_b = np.radians(2.0 * psi_b)
    return np.degrees(0.5 * np.arctan2(np.sin(two_a - two_b), np.cos(two_a - two_b)))


def plot_sym_map(x, y, data, title="", cbar_label="", cmap="coolwarm",
                 percentile=99.5, mask=None):
    # Symmetric (zero-centered) colorbar map. mask: True = include pixel.
    d   = np.where(mask, data, np.nan) if mask is not None else np.array(data, dtype=float)
    fin = d[np.isfinite(d)]
    if fin.size == 0:
        print(f"No finite pixels: {title}")
        return
    vmax = float(np.nanpercentile(np.abs(fin), percentile))
    ext  = [x.min(), x.max(), y.min(), y.max()]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(d, extent=ext, origin="lower", aspect="equal",
                   cmap=cmap, vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")
    plt.colorbar(im, ax=ax, label=cbar_label)
    plt.tight_layout()
    plt.show()


def summarize_angle_map(delta_psi, I_ref, db_min=-20, weight=None, label=""):
    # Summarize delta_psi over pixels where I_ref >= db_min dB.
    # weight: optional weight map (e.g. I_actual); if None, unweighted only.
    # Returns a dict of statistics, or None if no valid pixels.
    mask = make_db_mask(I_ref, db_min=db_min) & np.isfinite(delta_psi)
    if not np.any(mask):
        print(f"--- {label}: no pixels above {db_min:.0f} dB ---")
        return None
    d        = delta_psi[mask]
    n_total  = int(np.count_nonzero(np.isfinite(delta_psi)))
    n_sel    = int(mask.sum())
    frac_pix = n_sel / n_total
    u_mean   = float(np.mean(d))
    u_rms    = float(np.sqrt(np.mean(d**2)))
    max_abs  = float(np.max(np.abs(d)))
    result   = dict(db_min=db_min, pixels=n_sel, frac_pixels=frac_pix,
                    u_mean=u_mean, u_rms=u_rms, max_abs=max_abs)
    print(f"\n--- {label}: I_ref >= {db_min:.0f} dB ({n_sel} pixels, {frac_pix:.3%}) ---")
    if weight is not None:
        ww    = np.maximum(weight[mask], 0.0)
        ws    = float(ww.sum())
        w_all = float(np.nansum(np.maximum(weight, 0.0)))
        frac_wt = ws / w_all if w_all > 0 else np.nan
        if ws > 0:
            w_mean     = float(np.sum(ww * d) / ws)
            w_mean_abs = float(np.sum(ww * np.abs(d)) / ws)
            w_rms      = float(np.sqrt(np.sum(ww * d**2) / ws))
            result.update(frac_weight=frac_wt, w_mean=w_mean,
                          w_mean_abs=w_mean_abs, w_rms=w_rms)
            print(f"  fraction of total beam weight    = {frac_wt:.4f}")
            print(f"  weighted signed mean dpsi        = {w_mean:.8e} deg")
            print(f"  weighted mean |dpsi|             = {w_mean_abs:.8e} deg")
            print(f"  weighted RMS dpsi                = {w_rms:.8e} deg")
    print(f"  unweighted mean dpsi             = {u_mean:.8e} deg")
    print(f"  unweighted RMS dpsi              = {u_rms:.8e} deg")
    print(f"  max |dpsi|                       = {max_abs:.8e} deg")
    return result


In [ ]:
print("Loading files...")
x_xoff, y_xoff, F1_xoff, F2_xoff = load_grasp_grd(file_xoff)
x_yoff, y_yoff, F1_yoff, F2_yoff = load_grasp_grd(file_yoff)
x_xon,  y_xon,  F1_xon,  F2_xon  = load_grasp_grd(file_xon)
x_yon,  y_yon,  F1_yon,  F2_yon  = load_grasp_grd(file_yon)

for name, xi, yi in [("y_off", x_yoff, y_yoff),
                     ("x_on",  x_xon,  y_xon),
                     ("y_on",  x_yon,  y_yon)]:
    assert np.allclose(xi, x_xoff), f"x-grid mismatch: {name}"
    assert np.allclose(yi, y_xoff), f"y-grid mismatch: {name}"

x, y = x_xoff, y_xoff
print(f"Grid shape : {F1_xoff.shape}  |  x/y: {x.min():.3f} to {x.max():.3f} deg")
print("All four grids match: OK")

# Stokes for each run
Ix,   Qx,   Ux,   _ = compute_stokes(F1_xoff, F2_xoff)
Iy,   Qy,   Uy,   _ = compute_stokes(F1_yoff, F2_yoff)
Ixon, Qxon, Uxon, _ = compute_stokes(F1_xon,  F2_xon)
Iyon, Qyon, Uyon, _ = compute_stokes(F1_yon,  F2_yon)

print("\nPeak intensity per run:")
for lbl, I in [("x_off", Ix), ("y_off", Iy), ("x_on", Ixon), ("y_on", Iyon)]:
    print(f"  {lbl:8s}: {np.nanmax(I):.6e}")


## 2. Build actual and ideal calibrator Stokes maps

- **`S_actual`** = `0.5*(S_x_on + S_y_on)` — actual grid-on response to unpolarized input.
- **`S_ideal`**  = `0.5*S_y_off` — ideal y-selector through the no-grid telescope.
- **`S_unpol_off`** = `0.5*(S_x_off + S_y_off)` — baseline no-grid unpolarized (secondary reference).

> The ideal selector represents the case where the grid only picks out y-pol and leaves
> the telescope optics unchanged. It is the comparison target for the cleanest test.


In [ ]:
# Actual grid calibrator: dense-grid-on, unpolarized input
I_actual = 0.5 * (Ixon + Iyon)
Q_actual = 0.5 * (Qxon + Qyon)
U_actual = 0.5 * (Uxon + Uyon)

# Ideal y-selector through the no-grid telescope
I_ideal = 0.5 * Iy
Q_ideal = 0.5 * Qy
U_ideal = 0.5 * Uy

# Baseline no-grid unpolarized (secondary reference, not the primary comparison)
I_unpol_off = 0.5 * (Ix + Iy)
Q_unpol_off = 0.5 * (Qx + Qy)
U_unpol_off = 0.5 * (Ux + Uy)

print("Peak intensities:")
print(f"  I_actual     = {np.nanmax(I_actual):.6e}")
print(f"  I_ideal      = {np.nanmax(I_ideal):.6e}")
print(f"  I_unpol_off  = {np.nanmax(I_unpol_off):.6e}")
print(f"  peak ratio I_actual / I_ideal = {np.nanmax(I_actual) / np.nanmax(I_ideal):.6f}")


## 3. Compute key maps

**Per-pixel quantities:**
- `p_actual`, `psi_actual`: pol fraction and angle for the actual calibrator.
- `p_ideal`, `psi_ideal`: pol fraction and angle for the ideal selector.
- `delta_psi_grid_error = wrap(psi_actual - psi_ideal)`: the grid error angle — the core diagnostic.
- `dQ = Q_actual - Q_ideal`, `dU = U_actual - U_ideal`, `dI = I_actual - I_ideal`: Stokes residuals.
- Normalized: `dQ/I_ideal`, `dU/I_ideal`, `dI/I_ideal`.
- `delta_p = p_actual - p_ideal`.

> Angle maps (`psi_actual`, `psi_ideal`, `delta_psi_grid_error`) are only meaningful where `p` is not tiny.
> A mask combining the dB threshold with a minimum polarization fraction is applied.


In [ ]:
# Polarization fractions
p_actual = pol_fraction(Q_actual, U_actual, I_actual)
p_ideal  = pol_fraction(Q_ideal,  U_ideal,  I_ideal)
delta_p  = p_actual - p_ideal

# Polarization angles (full map, no masking yet)
psi_actual_full = pol_angle_deg(Q_actual, U_actual)
psi_ideal_full  = pol_angle_deg(Q_ideal,  U_ideal)

# Grid error angle (core diagnostic)
# Computed on full map; masking applied for display and statistics
delta_psi_grid_error = pol_angle_diff_deg(psi_actual_full, psi_ideal_full)

# Stokes residuals
dQ = Q_actual - Q_ideal
dU = U_actual - U_ideal
dI = I_actual - I_ideal

# Normalized residuals
Iref_safe = np.maximum(I_ideal, 1e-30)
dQ_norm   = dQ / Iref_safe
dU_norm   = dU / Iref_safe
dI_norm   = dI / Iref_safe

# Mask for angle maps: -20 dB AND p > 2%
# Angle maps are unreliable where p is very small.
mask20     = make_db_mask(I_ideal, db_min=DB_DEFAULT)
p_min      = 0.02
mask_angle = mask20 & (p_actual > p_min) & (p_ideal > p_min)

psi_actual = np.where(mask_angle, psi_actual_full, np.nan)
psi_ideal  = np.where(mask_angle, psi_ideal_full,  np.nan)
dpsi_masked = np.where(mask_angle, delta_psi_grid_error, np.nan)

print(f"dB mask ({DB_DEFAULT:.0f} dB): {mask20.sum()} pixels")
print(f"angle mask ({DB_DEFAULT:.0f} dB + p>{p_min:.2f}): {mask_angle.sum()} pixels")


In [ ]:
# --- Plots of key maps ---
ext = [x.min(), x.max(), y.min(), y.max()]
kw  = dict(extent=ext, origin="lower", aspect="equal")

# Pol fraction: actual vs ideal (plasma [0,1]), and their difference (coolwarm, symmetric)
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for ax, d, lbl in [(axs[0], np.where(mask20, p_actual, np.nan), "p_actual"),
                   (axs[1], np.where(mask20, p_ideal,  np.nan), "p_ideal")]:
    im = ax.imshow(d, cmap="plasma", vmin=0, vmax=1, **kw)
    ax.set_title(lbl); ax.set_xlabel("x [deg]")
    plt.colorbar(im, ax=ax, label="p")
# delta_p can be negative, so use a symmetric colormap
d_dp = np.where(mask20, delta_p, np.nan)
fin  = d_dp[np.isfinite(d_dp)]
lim  = float(np.nanpercentile(np.abs(fin), 99.5)) if fin.size > 0 else 1.0
im   = axs[2].imshow(d_dp, cmap="coolwarm", vmin=-lim, vmax=lim, **kw)
axs[2].set_title("delta_p = p_actual - p_ideal"); axs[2].set_xlabel("x [deg]")
plt.colorbar(im, ax=axs[2], label="delta p")
plt.suptitle("Polarization fraction  (above -20 dB mask)", fontsize=11)
plt.tight_layout(); plt.show()

# Grid error angle
plot_sym_map(x, y, dpsi_masked,
             title="Grid error angle  psi_actual - psi_ideal  (masked: -20dB, p>2%)",
             cbar_label="delta-psi [deg]", percentile=99.5)

# Normalized Stokes residuals
for d, lbl in [(np.where(mask20, dQ_norm, np.nan), "dQ / I_ideal"),
               (np.where(mask20, dU_norm, np.nan), "dU / I_ideal"),
               (np.where(mask20, dI_norm, np.nan), "dI / I_ideal")]:
    plot_sym_map(x, y, d, title=lbl, cbar_label="fraction", percentile=99.5)


## 4. Weighted/integrated statistics

**Weight choice:** `I_actual` (actual calibrator intensity).
This weights by the signal actually produced by the calibrator, so the
weighted mean answers: "What is the net angle error in the signal I actually measure?"

`I_ideal` is also reported as an alternative; it weights by the desired response.
Both are physically defensible; see separate rows in the table below.

**dB reference:** `I_ideal` (ideal selector) — the comparison target.
The mask is defined relative to the ideal beam so that the statistics directly quantify
the departure from ideal over the pixels that matter for an ideal calibrator.

> A small signed mean can hide structured positive and negative lobes.
> Always inspect the weighted RMS, max |dpsi|, and the contribution maps in Section 5.


In [ ]:
# Detailed summary at default dB threshold with both weight choices
print("Weighted by I_actual:")
res_actual = summarize_angle_map(
    delta_psi_grid_error, I_ref=I_ideal, db_min=DB_DEFAULT,
    weight=I_actual, label=f"psi_actual - psi_ideal at {DB_DEFAULT:.0f} dB"
)

print("\nWeighted by I_ideal:")
res_ideal_wt = summarize_angle_map(
    delta_psi_grid_error, I_ref=I_ideal, db_min=DB_DEFAULT,
    weight=I_ideal, label=f"psi_actual - psi_ideal at {DB_DEFAULT:.0f} dB"
)


In [ ]:
# Table over multiple dB thresholds
# Columns for both I_actual-weighted and I_ideal-weighted statistics.

db_cuts = [-10, -20, -30, -40, -50]
rows = []
for db_cut in db_cuts:
    mask = make_db_mask(I_ideal, db_min=db_cut) & np.isfinite(delta_psi_grid_error)
    if not np.any(mask):
        continue
    d      = delta_psi_grid_error[mask]
    n_tot  = int(np.count_nonzero(np.isfinite(delta_psi_grid_error)))
    n_sel  = int(mask.sum())
    row    = {"dB cut": db_cut, "pixels": n_sel, "frac pix": n_sel / n_tot}

    for wt_lbl, weight in [("wt=I_act", I_actual), ("wt=I_ideal", I_ideal)]:
        ww   = np.maximum(weight[mask], 0.0)
        ws   = float(ww.sum())
        wall = float(np.nansum(np.maximum(weight, 0.0)))
        if ws <= 0:
            continue
        wm   = float(np.sum(ww * d) / ws)
        wma  = float(np.sum(ww * np.abs(d)) / ws)
        wrms = float(np.sqrt(np.sum(ww * d**2) / ws))
        row[f"{wt_lbl} frac_wt"]    = ws / wall if wall > 0 else np.nan
        row[f"{wt_lbl} mean [deg]"] = wm
        row[f"{wt_lbl} |mean| [deg]"] = wma
        row[f"{wt_lbl} RMS [deg]"]  = wrms

    row["max|dpsi| [deg]"] = float(np.max(np.abs(d)))
    row["uw.mean [deg]"]   = float(np.mean(d))
    row["uw.RMS [deg]"]    = float(np.sqrt(np.mean(d**2)))
    rows.append(row)

df_table = pd.DataFrame(rows)
print("Grid error angle statistics by dB threshold  (dB ref = I_ideal):")
display(df_table)


## 5. Contribution maps

Two beam-weighted visualizations of `delta_psi_grid_error`:

**Relative map:** `(w / w_max) * delta_psi`
- Units are degrees. Values are suppressed by relative beam strength.
- Shows where the angle error matters most in a "beam-importance" sense.
- This is the better visualization for identifying spatially structured errors.

**Integral map:** `(w / sum(w)) * delta_psi` (deg/pixel)
- The literal per-pixel contribution to the beam-weighted mean.
- Sum of this map equals the weighted mean delta_psi.
- Shows whether the net mean is built up from coherent or canceling contributions.

> A relative map that looks structured even where the integral map is small means
> the angle errors cancel when integrated but do exist locally in the beam.


In [ ]:
# Weight by I_actual; use full finite map (not just -20 dB) so off-peak structure is visible
valid  = np.isfinite(delta_psi_grid_error) & np.isfinite(I_actual) & (I_actual > 0)
dpsi_v = delta_psi_grid_error
w_v    = np.where(valid, I_actual, 0.0)

wmax  = float(np.nanmax(w_v))
wsum  = float(np.nansum(w_v[valid]))

relative_map = np.where(valid, (w_v / wmax) * dpsi_v, np.nan)
integral_map = np.where(valid, (w_v / wsum) * dpsi_v, np.nan)

ext = [x.min(), x.max(), y.min(), y.max()]

# Relative map (units: deg, scaled by I/I_max)
fin  = relative_map[np.isfinite(relative_map)]
vmax = float(np.nanpercentile(np.abs(fin), 99.5)) if fin.size > 0 else 1.0
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(relative_map, extent=ext, origin="lower", aspect="equal",
               cmap="coolwarm", vmin=-vmax, vmax=vmax)
ax.set_title("Beam-weighted delta-psi  (I/I_max) * delta_psi")
ax.set_xlabel("x [deg]")
ax.set_ylabel("y [deg]")
plt.colorbar(im, ax=ax, label="(I/I_max) * dpsi [deg]")
plt.tight_layout()
plt.show()

# Integral map (units: deg/pixel contribution to weighted mean)
fin  = integral_map[np.isfinite(integral_map)]
vmax = float(np.nanpercentile(np.abs(fin), 99.5)) if fin.size > 0 else 1.0
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(integral_map, extent=ext, origin="lower", aspect="equal",
               cmap="coolwarm", vmin=-vmax, vmax=vmax)
ax.set_title("Per-pixel contribution to weighted mean  (I/sum I) * delta_psi")
ax.set_xlabel("x [deg]")
ax.set_ylabel("y [deg]")
plt.colorbar(im, ax=ax, label="(I/sum I) * dpsi [deg/pixel]")
plt.tight_layout()
plt.show()

# Verify: sum of integral map = weighted mean
map_sum    = float(np.nansum(integral_map))
wm_direct  = float(np.nansum(w_v[valid] * dpsi_v[valid]) / wsum)
print(f"Sum of integral map          = {map_sum:.12e} deg")
print(f"Direct weighted mean delta-psi = {wm_direct:.12e} deg")
print("These should match (small numerical difference OK).")


## 6. Final compact summary


In [ ]:
mask20 = make_db_mask(I_ideal, db_min=-20)

# Intensity ratios
pk_ratio  = np.nanmax(I_actual) / np.nanmax(I_ideal)
int_ratio = np.nansum(I_actual[mask20]) / np.nansum(I_ideal[mask20])

# Beam-weighted pol fractions
w_act  = I_actual * mask20; wsum_act = w_act.sum()
w_id   = I_ideal  * mask20; wsum_id  = w_id.sum()
p_act_mean = float(np.sum(w_act * p_actual) / wsum_act) if wsum_act > 0 else np.nan
p_id_mean  = float(np.sum(w_id  * p_ideal)  / wsum_id)  if wsum_id  > 0 else np.nan

# Grid error angle stats at -20 dB, weighted by I_actual
mask_s = mask20 & np.isfinite(delta_psi_grid_error)
d_s    = delta_psi_grid_error[mask_s]
ww_s   = np.maximum(I_actual[mask_s], 0.0); ws_s = ww_s.sum()
wm_s   = float(np.sum(ww_s * d_s) / ws_s)                     if ws_s > 0 else np.nan
wrms_s = float(np.sqrt(np.sum(ww_s * d_s**2) / ws_s))         if ws_s > 0 else np.nan
mabs_s = float(np.max(np.abs(d_s)))                            if d_s.size > 0 else np.nan

print("=" * 60)
print("FINAL SUMMARY: Dense grid vs ideal y-selector")
print("=" * 60)
print(f"\nIntensity comparison (actual / ideal):")
print(f"  Peak ratio I_actual / I_ideal          = {pk_ratio:.6f}")
print(f"  Integrated ratio above -20 dB          = {int_ratio:.6f}")
print(f"\nBeam-weighted pol fraction above -20 dB:")
print(f"  p_actual (wt=I_actual)                 = {p_act_mean:.6f}")
print(f"  p_ideal  (wt=I_ideal)                  = {p_id_mean:.6f}")
print(f"\nGrid error angle psi_actual - psi_ideal  (above -20 dB, wt=I_actual):")
print(f"  weighted mean dpsi                     = {wm_s:.6e} deg")
print(f"  weighted RMS  dpsi                     = {wrms_s:.6e} deg")
print(f"  max |dpsi|                             = {mabs_s:.6e} deg")


### Interpretation

**If weighted mean and RMS dpsi are small:**  
The dense grid acts approximately as an ideal y-polarization selector over the main beam.
The calibration signal closely matches what an ideal grid would produce through the no-grid telescope,
so the calibration is applicable to the normal grid-off instrument.

**If mean dpsi is small but RMS dpsi or residual maps are structured:**  
The grid does not produce a large global angle bias, but it does modify local polarized beam structure.
The signed contributions cancel on average but are non-zero locally.
In this case the calibration characterizes the spatially-averaged response well,
but local beam polarization structure in the grid-on case differs from the grid-off telescope.

**If both mean and RMS dpsi are large:**  
The grid changes the polarization angle response significantly.
The dense-grid calibration would characterize the "LAT with grid" rather than the normal telescope,
and cannot be directly applied as a calibration of the grid-off instrument.

> This notebook tests the dense-grid limit only. It does not address sparse-wire diffraction,
> wide-angle ground pickup, or behavior at other wire spacings.
> See companion notebooks for analyses at finite wire spacings.
